In [10]:
#%pip install torch matplotlib scipy
import torch
import matplotlib.pyplot as plt
from scipy.io import loadmat

In [ ]:
def init_params(nx, nh, ny):
    params = {}
    params["W1"] = torch.randn(nh, nx) * 0.3
    params["b1"] = torch.randn(1, nh) * 0.3
    params["W2"] = torch.randn(ny, nh) * 0.3
    params["b2"] = torch.randn(1, ny) * 0.3
    return params

In [4]:
def forward (params, X):
    outputs = {}
    outputs["X"] = X
    outputs["h_tilde"] = torch.mm(X, params["W1"].T) + params["b1"]
    outputs["h"] = torch.tanh(outputs["h_tilde"])
    outputs["y_tilde"] = torch.mm(outputs["h"], params["W2"].T) + params["b2"]
    exp_y = torch.exp(outputs["y_tilde"])
    outputs["y_hat"] = exp_y / torch.sum(exp_y, dim=1, keepdim=True)
    return outputs, outputs["y_hat"]

In [5]:
def loss_accuracy(y_hat, y):
    # adding 1e-12 to avoid log(0)
    L = -torch.mean(torch.sum(y * torch.log(y_hat + 1e-12), dim=1))

    _, indsY = torch.max(y, 1)
    _, indsYhat = torch.max(y_hat, 1)
    acc = torch.mean((indsY == indsYhat).float())

    return L, acc

In [6]:
def backward(params, outputs, y):
    grads = {}
    N = y.shape[0]

    grad_y_tilde = (outputs["y_hat"] - y) / N
    grads["W2"] = torch.mm(grad_y_tilde.T, outputs["h"])
    grads["b2"] = torch.sum(grad_y_tilde, dim=0, keepdim=True)

    grad_h = torch.mm(grad_y_tilde, params["W2"])
    grad_h_tilde = grad_h * (1 - outputs["h"] ** 2)
    grads["W1"] = torch.mm(grad_h_tilde.T, outputs["X"])
    grads["b1"] = torch.sum(grad_h_tilde, dim=0, keepdim=True)

    return grads

In [7]:
def sgd(params, grads, eta):
    for k in params:
        params[k] = params[k] - eta * grads[k]
    return params

In [11]:
def train(Xtrain, ytrain, Xtest, ytest, nh, Nbatch, eta, Nepoch):
    N, nx = Xtrain.shape
    ny = ytrain.shape[1]
    params = init_params(nx, nh, ny)
    curves = {"loss_train": [], "loss_test": [], "acc_train": [], "acc_test": []}

    for epoch in range(Nepoch):
        perm = torch.randperm(N)
        Xs, ys = Xtrain[perm], ytrain[perm]
        for j in range(N // Nbatch):
            Xb = Xs[j * Nbatch:(j + 1) * Nbatch]
            yb = ys[j * Nbatch:(j + 1) * Nbatch]
            outputs, yhat = forward(params, Xb)
            grads = backward(params, outputs, yb)
            params = sgd(params, grads, eta)

        Ltr, atr = loss_accuracy(forward(params, Xtrain)[1], ytrain)
        Lte, ate = loss_accuracy(forward(params, Xtest)[1], ytest)
        for k, v in zip(curves, [Ltr, Lte, atr, ate]):
            curves[k].append(v.item())
        print(f"epoch {epoch:3d} | train loss {Ltr:.4f} acc {atr:.3f} | test loss {Lte:.4f} acc {ate:.3f}")

    return params, curves